# Dimensionality Reduction And Clustering

**REQUIRED DAY 2**

## Load your checkpoint

Fresh kernel -- loading back the `adata` saved at the end of [06_normalization_and_feature_selection.ipynb](06_normalization_and_feature_selection.ipynb) (normalized, log-transformed, highly-variable genes flagged).

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("results/checkpoint_06_normalized.h5ad")
adata


## PCA, neighbors, UMAP

A cell-by-gene matrix with thousands of genes is too high-dimensional to cluster or visualize directly. The standard pipeline compresses it in stages. Look up scanpy's PCA function ([`sc.pp.pca`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pp.pca.html)) and run it with a deliberately generous `n_comps=50` — you'll choose how many to actually use just below. Then plot the variance-ratio curve ([`sc.pl.pca_variance_ratio`](https://scanpy.readthedocs.io/en/stable/generated/scanpy.pl.pca_variance_ratio.html), `n_pcs=50, log=True`).

In [ ]:
## Fill in: sc.pp.pca(adata, n_comps=50), then sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)



## How many PCs actually matter here?

`n_comps=50` above was deliberately generous — not a claim that all 50 are meaningful. The plot shows how much variance each additional PC explains; **there is no universally correct cutoff**, same as the clustering resolution below. Look for where the curve stops dropping sharply and flattens out (the "elbow") — the PCs past that point are contributing very little on top of what earlier ones already captured, mostly noise. Pick a number, then use it below.

In [ ]:
N_PCS = None  # set this based on where the curve above flattens out, then re-run
assert N_PCS is not None, "choose a value based on the variance-ratio plot above"

sc.pp.neighbors(adata, n_pcs=N_PCS)         # build a graph of each cell's nearest neighbors, using only your chosen PCs
sc.tl.umap(adata)                           # 2D layout for visualization only — not used for clustering itself
sc.pl.umap(adata)


PCA finds the axes of greatest variation; the neighbor graph captures which cells are transcriptionally similar; UMAP is a 2D projection for *looking at* that structure — it's a visualization, not the thing clustering actually operates on. Whatever `N_PCS` you chose feeds directly into the neighbor graph — too few and you lose real structure, too many and you're clustering on noise alongside signal.

## Clustering, and the parameter nobody can tell you the "right" value for

In [ ]:
import matplotlib.pyplot as plt

resolutions_to_try = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.5, 2.0]
n_clusters = []
for r in resolutions_to_try:
    sc.tl.leiden(adata, resolution=r, key_added=f"leiden_scan_{r}")
    n_clusters.append(adata.obs[f"leiden_scan_{r}"].nunique())

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(resolutions_to_try, n_clusters, marker="o")
ax.set_xlabel("resolution")
ax.set_ylabel("number of clusters found")


`resolution` controls how fine-grained the clusters are — higher resolution means more, smaller clusters. **There is no universally correct resolution.** The scan above shows cluster count as a function of resolution for *your* choice of PCs — it will often rise, then plateau for a stretch (a real stability region worth noticing), then rise again. Pick one resolution from your scan — not necessarily where it plateaus, that's a reasonable default argument but not the only valid one — and use it below:

In [ ]:
RESOLUTION = None  # pick a value based on the scan above, then re-run
assert RESOLUTION is not None, "choose a value based on the resolution scan above"

sc.tl.leiden(adata, resolution=RESOLUTION, key_added="leiden")
sc.pl.umap(adata, color="leiden")


The number itself isn't defensible on its own — **the resolution you end up using should be justified against something concrete**, like whether clusters separate along known marker genes (checked in [08_cell_type_annotation.ipynb](08_cell_type_annotation.ipynb)), or roughly matches the number of cell types you'd actually expect in this sample (PBMCs: T cells, B cells, NK cells, a couple of monocyte subtypes — very roughly 5-8 broad types), not left at whatever a tutorial's default happened to be.

## The other thing a cluster boundary might mean: not biology

If today's data had multiple lanes, batches, or donors, a cluster boundary that lines up suspiciously well with one of those variables — rather than with any marker gene — is a warning sign, not a discovery: could a batch, lane, or chemistry effect explain that boundary better than a real cell-type difference? Today's single shared sample has one lane pair (L001/L002) that were combined during alignment specifically so this isn't a live confound in your output — but the check is worth running as a habit, since it will matter the moment you work with real multi-batch data.

In [ ]:
sc.pl.umap(adata, color=["leiden", "total_counts", "pct_counts_mt"])


If a cluster boundary tracks `total_counts` or `pct_counts_mt` more cleanly than it tracks any biology you'd expect, that's a QC artifact bleeding into your clustering, not a cell type.

## Save your checkpoint

[08_cell_type_annotation.ipynb](08_cell_type_annotation.ipynb) loads this back in — from *this* point in the notebook, using today's single shared sample. The batch-effect section below is a separate, self-contained side exploration using an extra dataset; it does not change what gets saved here.

In [ ]:
adata.write_h5ad("results/checkpoint_07_clustered.h5ad")
print("Saved to results/checkpoint_07_clustered.h5ad")


## When it really is a batch effect: seeing one, and correcting it

The section above told you a real batch effect *would* show up as a cluster boundary that tracks a technical variable instead of biology — but today's single sample doesn't actually have one to show you. Let's manufacture a real one on purpose, using a second real, public PBMC dataset, so you see what this genuinely looks like and how to fix it. (This uses new variable names throughout — `adata` above and the checkpoint you just saved are untouched.)

`sc.datasets.pbmc3k()` is a classic, widely-used 10x dataset — PBMCs, but an older chemistry (10x v1, from ~2016) than today's shared sample (10x v3, much more recent). Different chemistry, different capture efficiency, different year: exactly the kind of technical difference that produces a real batch effect between two datasets of the same underlying cell types.

This normally downloads from the internet the first time it's called — but rather than have everyone's compute node do that independently mid-class, it's already been fetched once into today's shared data folder. Point `scanpy` at that shared cache before calling it, so this loads instantly from disk instead:

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # a couple of scanpy/anndata deprecation notices below are not worth the noise today

sc.settings.datasetdir = "/tscc/nfs/home/juf009/day2_shared_data/extra_datasets"
pbmc3k = sc.datasets.pbmc3k()          # reads from the shared cache above; would download on first use elsewhere
pbmc3k.var_names_make_unique()
pbmc3k.obs["batch"] = "pbmc3k_2016_v1chemistry"

today_sample = sc.read_h5ad("/tscc/nfs/home/juf009/day2_shared_data/counts/checkpoint.h5ad")  # today's sample, raw counts again
today_sample.var_names_make_unique()
today_sample.obs["batch"] = "today_2019_v3chemistry"

print(pbmc3k.shape, today_sample.shape)


Both datasets need to agree on which genes they're even talking about before you can combine them. Restrict both to the genes they have in common, then do the exact same QC filtering on each — you already know these functions from [05_loading_data_and_qc.ipynb](05_loading_data_and_qc.ipynb) (`sc.pp.filter_cells(adata, min_genes=200)`, `sc.pp.filter_genes(adata, min_cells=3)`).

In [ ]:
shared_genes = pbmc3k.var_names.intersection(today_sample.var_names)
print("genes in common:", len(shared_genes))

pbmc3k = pbmc3k[:, shared_genes].copy()
today_sample = today_sample[:, shared_genes].copy()

## Fill in: for each of pbmc3k and today_sample, run
##   sc.pp.filter_cells(adata, min_genes=200)
##   sc.pp.filter_genes(adata, min_cells=3)




Now combine them into one AnnData with [`sc.concat`](https://anndata.readthedocs.io/en/stable/generated/anndata.concat.html) (`join="inner"` keeps only the shared genes, which is already all of them here), normalize/log/HVG-select/scale/PCA/neighbors/UMAP on the *combined* object exactly as before, and plot the UMAP colored by `batch`.

In [ ]:
adata_batch = sc.concat([pbmc3k, today_sample], join="inner", label="batch_source")
adata_batch.obs_names_make_unique()

sc.pp.normalize_total(adata_batch, target_sum=1e4)
sc.pp.log1p(adata_batch)
sc.pp.highly_variable_genes(adata_batch, n_top_genes=2000, batch_key="batch")  # batch_key: pick HVGs that are variable in BOTH datasets, not just one
adata_batch = adata_batch[:, adata_batch.var.highly_variable].copy()
sc.pp.scale(adata_batch, max_value=10)
sc.tl.pca(adata_batch, n_comps=30)
sc.pp.neighbors(adata_batch)
sc.tl.umap(adata_batch)

sc.pl.umap(adata_batch, color="batch")


Look at that plot before reading on. If the two datasets form two largely separate blobs rather than mixing together — despite both being ordinary PBMC samples that should contain mostly the same cell types — that separation is the batch effect: a difference driven by *which dataset a cell came from*, not by *what cell type it is*.

## Fixing it with Harmony

[Harmony](https://www.nature.com/articles/s41592-019-0619-0) corrects for this by adjusting the PCA embedding so that cells cluster by shared biological signal rather than by dataset of origin, using the batch labels you provide as a guide.

One real thing worth knowing before you run this: **library APIs change, and wrappers around them can quietly fall out of sync.** The version of `harmonypy` installed here (2.0.0) returns its corrected embedding in a different orientation than scanpy's built-in wrapper (`sc.external.pp.harmony_integrate`) expects, which makes that wrapper throw a shape-mismatch error. Rather than paper over that, call `harmonypy` directly yourself, so you can see exactly what it returns and check it before using it — a smaller version of the same "verify, don't assume" habit from Lucas's tutorial and everywhere else in this bootcamp.

In [ ]:
import harmonypy

ho = harmonypy.run_harmony(adata_batch.obsm["X_pca"], adata_batch.obs, ["batch"])

## Fill in: check that ho.Z_corr.shape is (adata_batch.n_obs, 30) -- i.e., (cells, PCs) --
## before trusting it. If it's transposed instead, you'd need ho.Z_corr.T here instead.


adata_batch.obsm["X_pca_harmony"] = ho.Z_corr


Now rebuild the neighbor graph and UMAP using the Harmony-corrected embedding (`use_rep="X_pca_harmony"`) instead of the default `X_pca`, and plot it colored by `batch` again.

In [ ]:
## Fill in:
## sc.pp.neighbors(adata_batch, use_rep="X_pca_harmony")
## sc.tl.umap(adata_batch)
## sc.pl.umap(adata_batch, color="batch")




Compare the two `batch`-colored UMAPs. After correction, the two datasets should mix much more thoroughly — cells are now grouping by whatever biological structure Harmony could find in common between them, rather than by which dataset produced them.

Two things worth sitting with, not just running past:

- **Correction is not automatically correct.** Harmony was told to remove the effect of `batch` — if `batch` had accidentally also captured a real biological difference (say, if one dataset were healthy donors and the other were a disease cohort), Harmony would happily remove *that* signal too, and you'd have quietly erased the exact thing you meant to study. Batch correction should be justified by what `batch` actually represents technically, the same way a QC threshold or a clustering resolution needs to be justified — not applied by default because a tutorial did it.
- **This is genuinely the same idea Aim 1-style analyses run into constantly**: is a difference between two samples/conditions a real biological difference, or a technical one that needs to be corrected out first? You just watched both versions of that question happen in the same UMAP.

## Practice

State the PC count and the resolution you chose earlier, and why, in one sentence each -- the two real judgment calls the main part of this notebook asked you to make. Then, for the batch-correction side exploration: in one sentence, what specifically would make you *not* trust a Harmony correction on a real dataset (i.e., what would `batch` need to be confounded with for correction to be the wrong move)?

## Further reading

- [Single-cell best practices — Dimensionality Reduction](https://www.sc-best-practices.org/preprocessing_visualization/dimensionality_reduction.html)
- [Single-cell best practices — Clustering](https://www.sc-best-practices.org/cellular_structure/clustering.html)
- [Single-cell best practices — Batch effects and integration](https://www.sc-best-practices.org/cellular_structure/integration.html)
- [Harmony paper (Korsunsky et al., Nat Methods 2019)](https://www.nature.com/articles/s41592-019-0619-0)
- [Seurat's `RunHarmony`](https://satijalab.org/seurat/reference/runharmony) — the R/Seurat equivalent of this notebook's batch-correction section.